<a href="https://colab.research.google.com/github/AshokGit544/Enterprise-Finance-KnowledgeOps-Copilot/blob/main/Enterprise_Finance_KnowledgeOps_Copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pandas numpy pyspark sentence-transformers faiss-cpu langchain langchain-community transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
!pip -q install requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.8/64.8 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.1 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.


In [4]:
import os
import json
import random
import zipfile
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

In [5]:
random.seed(42)
np.random.seed(42)

PROJECT_DIR = Path("Enterprise-Finance-KnowledgeOps-Copilot")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
VECTOR_DIR = PROJECT_DIR / "data" / "vector_store"
OUTPUT_DIR = PROJECT_DIR / "data" / "output"
DAG_DIR = PROJECT_DIR / "dags"
NOTEBOOK_DIR = PROJECT_DIR / "notebooks"
CONFIG_DIR = PROJECT_DIR / "config"

for folder in [RAW_DIR, PROCESSED_DIR, VECTOR_DIR, OUTPUT_DIR, DAG_DIR, NOTEBOOK_DIR, CONFIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders created successfully.")
print("Project root:", PROJECT_DIR.resolve())

Project folders created successfully.
Project root: /content/Enterprise-Finance-KnowledgeOps-Copilot


In [7]:
vendors = pd.DataFrame([
    ["V001", "Alpha Energy Services", "Maintenance", "NY", "Active"],
    ["V002", "BlueGrid Technologies", "IT Services", "CT", "Active"],
    ["V003", "Northwind Electric", "Electrical Components", "MA", "Active"],
    ["V004", "GreenVolt Contractors", "Field Services", "PA", "Active"],
    ["V005", "Prime Utility Logistics", "Logistics", "NJ", "Review"],
    ["V006", "CoreAxis Industrial Supply", "Industrial Supply", "MI", "Active"]
], columns=["vendor_id", "vendor_name", "vendor_category", "vendor_state", "vendor_status"])


gl_accounts = pd.DataFrame([
    ["400100", "Revenue - Energy Delivery", "Revenue"],
    ["500100", "Opex - Maintenance", "Expense"],
    ["500200", "Opex - IT Services", "Expense"],
    ["500300", "Opex - Contractors", "Expense"],
    ["500400", "Opex - Logistics", "Expense"],
    ["500500", "Opex - Field Operations", "Expense"],
    ["110100", "Accounts Payable", "Liability"]
], columns=["gl_account", "gl_account_name", "gl_type"])


cost_centers = pd.DataFrame([
    ["CC100", "Transmission Operations", "Operations"],
    ["CC110", "Distribution Maintenance", "Operations"],
    ["CC120", "Substation Engineering", "Engineering"],
    ["CC130", "Customer Support", "Customer Ops"],
    ["CC140", "Finance Operations", "Finance"],
    ["CC150", "Grid Modernization", "Engineering"]
], columns=["cost_center_id", "cost_center_name", "department"])


finance_policies = pd.DataFrame([
    ["POL001", "Invoice Policy", "Invoices above 40000 USD require finance manager approval before payment release.", "High"],
    ["POL002", "Vendor Compliance Policy", "Invoices from vendors in review status must be flagged for manual compliance verification.", "High"],
    ["POL003", "Duplicate Control Policy", "Invoices with same vendor, amount, and close invoice date must be checked for duplicate risk.", "Medium"],
    ["POL004", "Missing Data Policy", "Invoices with missing vendor, missing description, or invalid amount must be blocked from downstream approval workflows.", "High"],
    ["POL005", "Late Payment Policy", "Payments delayed beyond 30 days from posting date should be investigated for operational or financial control gaps.", "Medium"]
], columns=["policy_id", "policy_name", "policy_text", "severity"])


vendors.to_csv(RAW_DIR / "vendors.csv", index=False)
gl_accounts.to_csv(RAW_DIR / "gl_accounts.csv", index=False)
cost_centers.to_csv(RAW_DIR / "cost_centers.csv", index=False)
finance_policies.to_csv(RAW_DIR / "finance_policies.csv", index=False)

print("Master data created successfully.")
print("Files saved:")
print("-", RAW_DIR / "vendors.csv")
print("-", RAW_DIR / "gl_accounts.csv")
print("-", RAW_DIR / "cost_centers.csv")
print("-", RAW_DIR / "finance_policies.csv")

Master data created successfully.
Files saved:
- Enterprise-Finance-KnowledgeOps-Copilot/data/raw/vendors.csv
- Enterprise-Finance-KnowledgeOps-Copilot/data/raw/gl_accounts.csv
- Enterprise-Finance-KnowledgeOps-Copilot/data/raw/cost_centers.csv
- Enterprise-Finance-KnowledgeOps-Copilot/data/raw/finance_policies.csv


In [8]:
invoice_rows = []
payment_rows = []

start_date = datetime(2024, 1, 1)

descriptions = [
    "transformer repair service",
    "sap fico support work",
    "field contractor invoice",
    "substation maintenance work",
    "it platform support",
    "logistics support for equipment",
    "distribution network inspection",
    "grid modernization consulting",
    "vendor maintenance support"
]

for i in range(1, 151):
    vendor = vendors.sample(1).iloc[0]
    gl = gl_accounts[gl_accounts["gl_type"].isin(["Expense", "Revenue"])].sample(1).iloc[0]
    cc = cost_centers.sample(1).iloc[0]

    posting_date = start_date + timedelta(days=random.randint(0, 365))
    invoice_date = posting_date - timedelta(days=random.randint(1, 10))
    amount = round(random.uniform(1000, 60000), 2)

    source_doc_id = f"SRC{i:04d}"
    invoice_id = f"INV{i:05d}"

    invoice_rows.append([
        invoice_id,
        source_doc_id,
        vendor["vendor_id"],
        gl["gl_account"],
        cc["cost_center_id"],
        invoice_date.date().isoformat(),
        posting_date.date().isoformat(),
        amount,
        random.choice(descriptions)
    ])

    payment_delay = random.randint(5, 45)
    payment_date = posting_date + timedelta(days=payment_delay)

    payment_rows.append([
        f"PAY{i:05d}",
        source_doc_id,
        invoice_id,
        payment_date.date().isoformat(),
        round(amount * random.uniform(0.95, 1.00), 2)
    ])

invoices = pd.DataFrame(invoice_rows, columns=[
    "invoice_id", "source_doc_id", "vendor_id", "gl_account", "cost_center_id",
    "invoice_date", "posting_date", "amount_usd", "description"
])

payments = pd.DataFrame(payment_rows, columns=[
    "payment_id", "source_doc_id", "invoice_id", "payment_date", "paid_amount_usd"
])

# Add enterprise-style data quality and control issues
invoices.loc[3, "vendor_id"] = None
invoices.loc[7, "amount_usd"] = -500
invoices.loc[10, "description"] = None

# duplicate-like scenario
invoices.loc[20, ["vendor_id", "amount_usd", "invoice_date", "description"]] = [
    invoices.loc[21, "vendor_id"],
    invoices.loc[21, "amount_usd"],
    invoices.loc[21, "invoice_date"],
    invoices.loc[21, "description"]
]

invoices.to_csv(RAW_DIR / "invoices.csv", index=False)
payments.to_csv(RAW_DIR / "payments.csv", index=False)

print("Transaction data created successfully.")
print("Invoices shape:", invoices.shape)
print("Payments shape:", payments.shape)
display(invoices.head())
display(payments.head())

Transaction data created successfully.
Invoices shape: (150, 9)
Payments shape: (150, 5)


,invoice_id,source_doc_id,vendor_id,gl_account,cost_center_id,invoice_date,posting_date,amount_usd,description
0,INV00001,SRC0001,V001,500300,CC140,2024-11-21,2024-11-23,2475.63,it platform support
1,INV00002,SRC0002,V001,500200,CC150,2024-02-13,2024-02-22,6129.39,distribution network inspection
2,INV00003,SRC0003,V001,400100,CC100,2024-04-17,2024-04-21,30815.96,transformer repair service
3,INV00004,SRC0004,None,500500,CC150,2024-11-19,2024-11-28,25751.67,grid modernization consulting
4,INV00005,SRC0005,V004,500400,CC140,2024-01-01,2024-01-04,42190.22,logistics support for equipment


,payment_id,source_doc_id,invoice_id,payment_date,paid_amount_usd
0,PAY00001,SRC0001,INV00001,2024-12-13,2379.48
1,PAY00002,SRC0002,INV00002,2024-02-29,5832.05
2,PAY00003,SRC0003,INV00003,2024-05-31,29581.53
3,PAY00004,SRC0004,INV00004,2025-01-09,24822.28
4,PAY00005,SRC0005,INV00005,2024-01-26,40408.69


In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lit, concat_ws, when, trim, upper, datediff, to_date
)

spark = SparkSession.builder.appName("EnterpriseFinanceKnowledgeOpsCopilot").getOrCreate()

base_path = str(RAW_DIR)
processed_path = str(PROCESSED_DIR)
output_path = str(OUTPUT_DIR)

vendors_spark = spark.read.option("header", True).csv(f"{base_path}/vendors.csv")
gl_accounts_spark = spark.read.option("header", True).csv(f"{base_path}/gl_accounts.csv")
cost_centers_spark = spark.read.option("header", True).csv(f"{base_path}/cost_centers.csv")
finance_policies_spark = spark.read.option("header", True).csv(f"{base_path}/finance_policies.csv")
invoices_spark = spark.read.option("header", True).csv(f"{base_path}/invoices.csv")
payments_spark = spark.read.option("header", True).csv(f"{base_path}/payments.csv")

# Standardization
vendors_spark = vendors_spark.withColumn("vendor_name_clean", upper(trim(col("vendor_name"))))
invoices_spark = invoices_spark.withColumn("description_clean", upper(trim(col("description"))))

# Common business key
invoices_spark = invoices_spark.withColumn(
    "common_business_key",
    concat_ws("_", col("vendor_id"), col("cost_center_id"), col("gl_account"))
)

# Integrated finance view
integrated = (
    invoices_spark
    .join(vendors_spark, on="vendor_id", how="left")
    .join(gl_accounts_spark, on="gl_account", how="left")
    .join(cost_centers_spark, on="cost_center_id", how="left")
    .join(
        payments_spark.select("invoice_id", "payment_id", "payment_date", "paid_amount_usd"),
        on="invoice_id",
        how="left"
    )
)

# Date conversion for control checks
integrated = (
    integrated
    .withColumn("posting_date_dt", to_date(col("posting_date")))
    .withColumn("payment_date_dt", to_date(col("payment_date")))
    .withColumn("invoice_date_dt", to_date(col("invoice_date")))
    .withColumn("payment_delay_days", datediff(col("payment_date_dt"), col("posting_date_dt")))
)

# Finance risk / control flags
integrated = (
    integrated
    .withColumn(
        "high_value_flag",
        when(col("amount_usd").cast("double") > 40000, lit("Y")).otherwise(lit("N"))
    )
    .withColumn(
        "vendor_review_flag",
        when(col("vendor_status") == "Review", lit("Y")).otherwise(lit("N"))
    )
    .withColumn(
        "late_payment_flag",
        when(col("payment_delay_days") > 30, lit("Y")).otherwise(lit("N"))
    )
)

# Data quality checks
dq = integrated.select(
    "invoice_id",
    when(col("vendor_id").isNull(), lit("Missing vendor_id"))
    .when(col("amount_usd").cast("double") <= 0, lit("Invalid amount"))
    .when(col("description").isNull(), lit("Missing description"))
    .otherwise(lit(None)).alias("issue_reason")
).filter(col("issue_reason").isNotNull())

# Retrieval-ready finance narrative
retrieval_docs = integrated.withColumn(
    "retrieval_text",
    concat_ws(
        " ",
        lit("Invoice"),
        col("invoice_id"),
        lit("Vendor"),
        col("vendor_name"),
        lit("Vendor Status"),
        col("vendor_status"),
        lit("Cost Center"),
        col("cost_center_name"),
        lit("GL Account"),
        col("gl_account_name"),
        lit("Amount"),
        col("amount_usd"),
        lit("Description"),
        col("description"),
        lit("High Value Flag"),
        col("high_value_flag"),
        lit("Late Payment Flag"),
        col("late_payment_flag")
    )
)

embedding_input = retrieval_docs.select(
    "invoice_id",
    "common_business_key",
    "vendor_id",
    "vendor_name",
    "cost_center_id",
    "cost_center_name",
    "gl_account",
    "gl_account_name",
    "amount_usd",
    "retrieval_text"
)

print("Integrated finance view:")
integrated.show(5, truncate=False)

print("Data quality issues:")
dq.show(truncate=False)

print("Embedding input:")
embedding_input.show(5, truncate=False)

Integrated finance view:
+----------+--------------+----------+---------+-------------+------------+------------+----------+-------------------------------+-------------------------------+-------------------+---------------------+---------------+------------+-------------+---------------------+-------------------------+-------+-----------------------+-----------+----------+------------+---------------+---------------+---------------+---------------+------------------+---------------+------------------+-----------------+
|invoice_id|cost_center_id|gl_account|vendor_id|source_doc_id|invoice_date|posting_date|amount_usd|description                    |description_clean              |common_business_key|vendor_name          |vendor_category|vendor_state|vendor_status|vendor_name_clean    |gl_account_name          |gl_type|cost_center_name       |department |payment_id|payment_date|paid_amount_usd|posting_date_dt|payment_date_dt|invoice_date_dt|payment_delay_days|high_value_flag|vendor_revi

In [10]:
integrated_pd = integrated.toPandas()
dq_pd = dq.toPandas()
embedding_input_pd = embedding_input.toPandas()
finance_policies_pd = finance_policies_spark.toPandas()

integrated_pd.to_csv(PROCESSED_DIR / "integrated_finance_view.csv", index=False)
dq_pd.to_csv(OUTPUT_DIR / "data_quality_issues.csv", index=False)
embedding_input_pd.to_csv(PROCESSED_DIR / "embedding_input.csv", index=False)
finance_policies_pd.to_csv(PROCESSED_DIR / "finance_policies.csv", index=False)

print("Processed files saved successfully.\n")
print("Saved files:")
for p in sorted(PROCESSED_DIR.glob("*.csv")):
    print("-", p.name)
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", p.name)

Processed files saved successfully.

Saved files:
- embedding_input.csv
- finance_policies.csv
- integrated_finance_view.csv
- data_quality_issues.csv


In [11]:
from sentence_transformers import SentenceTransformer

embedding_input_df = pd.read_csv(PROCESSED_DIR / "embedding_input.csv")
finance_policies_df = pd.read_csv(PROCESSED_DIR / "finance_policies.csv")

finance_record_docs = []
for _, row in embedding_input_df.iterrows():
    finance_record_docs.append({
        "doc_id": f"FIN_{row['invoice_id']}",
        "doc_type": "finance_record",
        "source_id": row["invoice_id"],
        "common_business_key": row["common_business_key"],
        "text": row["retrieval_text"],
        "metadata": {
            "invoice_id": row["invoice_id"],
            "vendor_id": row["vendor_id"],
            "vendor_name": row["vendor_name"],
            "cost_center_id": row["cost_center_id"],
            "cost_center_name": row["cost_center_name"],
            "gl_account": row["gl_account"],
            "gl_account_name": row["gl_account_name"],
            "amount_usd": row["amount_usd"]
        }
    })

policy_docs = []
for _, row in finance_policies_df.iterrows():
    policy_docs.append({
        "doc_id": f"POL_{row['policy_id']}",
        "doc_type": "finance_policy",
        "source_id": row["policy_id"],
        "common_business_key": "POLICY",
        "text": f"Policy Name: {row['policy_name']}. Policy Text: {row['policy_text']}. Severity: {row['severity']}.",
        "metadata": {
            "policy_id": row["policy_id"],
            "policy_name": row["policy_name"],
            "severity": row["severity"]
        }
    })

all_docs = finance_record_docs + policy_docs

documents_df = pd.DataFrame([
    {
        "doc_id": d["doc_id"],
        "doc_type": d["doc_type"],
        "source_id": d["source_id"],
        "common_business_key": d["common_business_key"],
        "text": d["text"],
        "metadata_json": json.dumps(d["metadata"])
    }
    for d in all_docs
])

documents_df.to_csv(PROCESSED_DIR / "rag_documents.csv", index=False)

print("RAG documents prepared successfully.")
print("Total documents:", len(documents_df))
display(documents_df.head())

RAG documents prepared successfully.
Total documents: 155


,doc_id,doc_type,source_id,common_business_key,text,metadata_json
0,FIN_INV00001,finance_record,INV00001,V001_CC140_500300,Invoice INV00001 Vendor Alpha Energy Services ...,"{""invoice_id"": ""INV00001"", ""vendor_id"": ""V001""..."
1,FIN_INV00002,finance_record,INV00002,V001_CC150_500200,Invoice INV00002 Vendor Alpha Energy Services ...,"{""invoice_id"": ""INV00002"", ""vendor_id"": ""V001""..."
2,FIN_INV00003,finance_record,INV00003,V001_CC100_400100,Invoice INV00003 Vendor Alpha Energy Services ...,"{""invoice_id"": ""INV00003"", ""vendor_id"": ""V001""..."
3,FIN_INV00004,finance_record,INV00004,CC150_500500,Invoice INV00004 Vendor Vendor Status Cost Cen...,"{""invoice_id"": ""INV00004"", ""vendor_id"": NaN, ""..."
4,FIN_INV00005,finance_record,INV00005,V004_CC140_500400,Invoice INV00005 Vendor GreenVolt Contractors ...,"{""invoice_id"": ""INV00005"", ""vendor_id"": ""V004""..."
